In [11]:
# Routing 路由模式
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.graph import MessagesState, StateGraph
from langgraph.constants import START, END
from typing import Literal
from dotenv import load_dotenv
from IPython.core.display import Image
# 定义LLM
load_dotenv()
model = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

# 定义State
class IntentState(MessagesState):
    intent: str
    data: str

# 定义Node
def llm_node(state: IntentState) -> dict:
    question = f"""
        请根据用户问题，选择合适的数据源路由，你只能回答relational、graph、vector三者之一，不要输出多余内容
        用户问题:{state["messages"][-1].text}
    """
    intent = model.invoke(question).text
    return {"intent": intent}

def relational_node(state: IntentState) -> dict:
    return {"data": "关系型数据库"}

def graph_node(state: IntentState) -> dict:
    return {"data": "图数据库"}

def vector_node(state: IntentState) -> dict:
    return {"data": "向量数据库"}

# 定义Router
def router(state: IntentState) -> Literal["relational", "graph", "vector"]:
    return state["intent"]

# 定义Graph
graph_builder = StateGraph(IntentState)
graph_builder.add_node("llm", llm_node)
graph_builder.add_node("relational", relational_node)
graph_builder.add_node("graph", graph_node)
graph_builder.add_node("vector", vector_node)
graph_builder.add_edge(START, "llm")
graph_builder.add_conditional_edges("llm", router)
graph_builder.add_edge("relational", END)
graph_builder.add_edge("graph", END)
graph_builder.add_edge("vector", END)
graph = graph_builder.compile()

# 执行Graph
graph.invoke({"messages": [HumanMessage("汽车名称为大众的车价格多少钱")]})

{'messages': [HumanMessage(content='汽车名称为大众的车价格多少钱', additional_kwargs={}, response_metadata={}, id='2d4c4246-7efe-47fe-8159-c2804d760ce8')],
 'intent': 'relational',
 'data': '关系型数据库'}